# Seat Sweeper: from starting data to a working agent app

**Participant workbook | CoCo Desktop | 180-minute target**

Build an opportunity dashboard from synthetic software-seat data, deploy it, make a visible change, connect a named Cortex Agent, then watch a new PDF change an answer without redeploying.

**The route:** Git kit → seed + staged PDFs → dynamic tables → unchanged app deployment → visible change + redeployment → direct agent tests → Ask integration + third deployment → Zoom PDF arrival.

This repository contains only the V2 workshop. Use this notebook and the [V2 launch page](README.md).

## How to work

- **Notebook / Python:** run only the small preflight and configuration cells. Select an available local Python 3 kernel; no packages are installed automatically. Do not use Run All as a deployment mechanism.
- **Terminal:** copy Bash blocks into the CoCo terminal from the repository root. These are not notebook code cells. Python variables do not set terminal variables.
- **CoCo prompt:** paste the complete `text` block into chat. Replace `<LAB_DB>`, `<LAB_WH>` and `<LAB_ROLE>` with your settings. Review the proposed plan and approve writes deliberately.
- **Evidence:** edit the evidence lines as you finish. Record failures rather than checking a box without proof. Clear outputs and personal settings before sharing your notebook.

**Safety:** use your fresh participant account with ACCOUNTADMIN. Never use the `rehearsal` target, a maintainer connection, or the original demo database. Setup, refreshes, PDF parsing, Search, Agent questions and app builds may incur charges. These are fictional agreements, not legal advice or actual vendor terms.

**Validation status:** this is a rehearsal kit; publication is not evidence that a fresh-account run passed. Record the cloned commit and use the facilitator's pinned revision for a group workshop. Allocations below are targets, not measured timings.

| Segment | Minutes |
|---|---:|
| Git, tools, identity | 15 |
| Starting data and staged PDFs | 15 |
| Pipelines and inspection | 30 |
| Starter deploy | 20 |
| Personal change and redeploy | 15 |
| Break | 10 |
| Build and test agent | 20 |
| Ask integration and deploy | 30 |
| PDF arrival | 15 |
| Evidence and costs | 10 |

## 1. Get the kit and confirm your account

**Goal:** work from the same reviewed release in your own account.

If you already cloned the kit, verify and record its revision. Otherwise follow the [Git launch instructions](README.md) for the V2-only repository. For a group workshop, check out the facilitator's approved tag or commit. Open the repository folder in CoCo Desktop, then this notebook.

**Terminal / read-only:**

```bash
git rev-parse HEAD
git status --short
python3 --version
node --version
npm --version
snow --version
snow app --help
cortex agent-studio --help
```

Use Node.js 22 LTS and Python 3.10+ (the validator uses modern pathlib). Snowflake CLI 3.29.0 was used during development; confirm `snow app` and version-2 manifests are supported. Stop and resolve missing tools with your facilitator rather than silently upgrading or installing packages. Agent deployment also requires the `cortex agent-studio` command used by the supplied helper; verify it is available before starting. The local Python kernel and terminal Python can differ.

**Notebook / local-only:** run the next cell to locate kit files and verify all PDF checksums. It makes no Snowflake calls and changes no files.

**Success:** correct Git revision, clean starting checkout, required files found, eleven PDFs verified. A dirty checkout is not automatically wrong, but inspect it before proceeding.

**Recovery:** missing notebook/assets means wrong or incomplete release; stop and get the complete kit. For a missing kernel, select a local Python kernel through CoCo Desktop before running cells.

**Evidence:** release commit: ______; account identity confirmed: ______.

In [ ]:
from pathlib import Path
import runpy
import shutil
import sys

if sys.version_info < (3, 10):
    raise RuntimeError('Use a local Python 3.10+ kernel.')
current = Path.cwd().resolve()
repo_root = next((candidate for candidate in [current, *current.parents]
                  if (candidate / 'sql/v2/setup.sh').is_file()
                  and (candidate / 'v2/starter-app/app.yml').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open the cloned repository folder in CoCo Desktop first.')
required = ['v2/START_HERE.ipynb', 'v2/starter-app/package-lock.json',
            'sql/v2/checkpoint_vendor_opportunity.sql',
            'v2/checkpoints/apply_completed_ask.sh', 'scripts/contract_pipeline.py']
missing = [name for name in required if not (repo_root / name).is_file()]
if missing:
    raise RuntimeError(f'Incomplete workshop revision: {missing}')
pipeline_module = runpy.run_path(str(repo_root / 'scripts/contract_pipeline.py'))
pdfs = pipeline_module['manifest_rows']('all')
print(f'Kit found; {len(pdfs)} PDF checksums verified.')
print('Kernel Python:', sys.version.split()[0])
print('Terminal tools visible to kernel:', {tool: bool(shutil.which(tool))
      for tool in ['git', 'python3', 'node', 'npm', 'snow', 'cortex']})

### Set connection once, then verify before writes

Use a named Snowflake CLI connection configured to your fresh account. Authentication stays in your local Snowflake configuration, never this notebook. Do not paste a password, PAT or private key into a cell or chat.

**Notebook / local-only:** replace the three placeholders below. The cell only prints shell-safe exports and a read-only identity command; it does not execute them. These settings use the supplied attendee namespace. Keep it unchanged for this workshop.

**Terminal:** copy the printed block into the repository-root terminal. Confirm the returned organization/account and user exactly match your expected values and that ACCOUNTADMIN is active. Do not run setup if they differ. Tell CoCo this named connection and the confirmed namespace; the connection selected in chat is not automatically the same as the terminal's.

If no named connection exists, use `snow connection add --help` and your facilitator's approved authentication method. Do not put credentials in Git.

**Recovery:** wrong identity is a connection problem, not a reason to change target names. Correct it and rerun the identity check before any mutation.

In [ ]:
import shlex

connection_name = 'REPLACE_WITH_YOUR_NAMED_CONNECTION'
expected_account = 'REPLACE_WITH_ORGANIZATION.ACCOUNT_NAME'
expected_user = 'REPLACE_WITH_YOUR_USER'
if any(value.startswith('REPLACE_') for value in
       [connection_name, expected_account, expected_user]):
    raise ValueError('Replace the three settings with your own account values first.')
settings = {'LAB_CONNECTION': connection_name, 'LAB_DB': 'SEAT_SWEEPER_LAB',
            'LAB_WH': 'SEAT_SWEEPER_WH', 'LAB_ROLE': 'SEAT_SWEEPER_APP'}
print('\n'.join(f'export {key}={shlex.quote(value)}' for key, value in settings.items()))
print('snow sql --connection "$LAB_CONNECTION" --role ACCOUNTADMIN -q '
      + shlex.quote("SELECT CURRENT_ORGANIZATION_NAME() || '.' || CURRENT_ACCOUNT_NAME() AS account, CURRENT_USER() AS user_name, CURRENT_ROLE() AS role"))
print(f'Compare the returned account to {expected_account} and user to {expected_user}.')
print('STOP if either differs. This cell did not connect or deploy anything.')

## 2. Deploy starting data from the Git kit

**Goal:** create synthetic raw data and stage ten baseline PDFs. Keep Zoom out.

**Action:** review [setup](../sql/v2/setup.sh), [object setup](../sql/v2/00_setup.sql), [seed](../sql/v2/01_seed.sql) and [default names](../sql/v2/names.default.env) with CoCo. Setup creates the lab namespace, warehouse, role and build integration; it loads synthetic employees, contracts, assets and assignments, then stages PDFs. It does **not** build downstream dynamic tables, Search or the agent. Contract clauses are not seeded as text.

Approve this billable setup only after confirming identity and that the namespace is unused. The shell script displays identity but does not pause for your confirmation.

**Terminal / writes, after review:**

```bash
bash sql/v2/setup.sh "$LAB_CONNECTION"
```

Open the synthetic [Figma PDF](contracts/baseline/figma-enterprise-v1.pdf) and note a clause to verify later. The [manifest](contracts/manifest.json) defines immutable document paths and SHA-256 hashes.

**Success:** setup finishes without error; four raw tables are populated; ten baseline PDFs are staged; Zoom remains only in the kit. Ask CoCo for a read-only source-count and stage/manifest check using your named connection.

**Recovery:** setup skips seeding when EMPLOYEES is nonempty. That is not proof all four tables are complete. If interrupted, inspect partial state before retrying; do not delete the database or overwrite documents. Get facilitator help for an object collision or an unavailable account capability.

**Evidence:** raw row counts: ______; baseline staged: ______; Figma clause: ______.

## 3. Build the structured dynamic tables with CoCo

**Goal:** assignments → enrichment → vendor opportunity → app-compatible view.

**CoCo prompt / plan first:** replace the angle-bracket settings, paste, review, then approve the proposed writes. [Canonical prompt](coco-prompts/build-pipeline.md).

<!-- prompt: coco-prompts/build-pipeline.md#1 -->
```text
Using my confirmed lab connection, inspect <LAB_DB>.RAW.EMPLOYEES, ASSETS, CONTRACTS and EMPLOYEE_ASSETS, including columns, row counts, unique keys and missing employee/asset references. Read the dynamic-tables skill. Plan two explicit INCREMENTAL dynamic tables: <LAB_DB>.APP.DT_SEAT_ASSIGNMENTS enriches assignments with employee department, matched contract_id, vendor name, has_contract, status, last_active_date and monthly_cost_per_seat; <LAB_DB>.APP.DT_VENDOR_OPPORTUNITY aggregates eligible assignments. Use <LAB_WH>, DOWNSTREAM for enrichment, a 1-minute target lag for the aggregate, and ON_CREATE initialization. Explain ongoing refresh costs.

Eligible means status = 'Inactive', last_active_date at least 30 days before CURRENT_DATE(), and positive monthly_cost_per_seat. Keep CURRENT_DATE in WHERE, not SELECT; do not silently use FULL/AUTO or change the business rule. Eligibility reflects the last successful refresh, not the browser's current clock.

Preserve grain (matched contract_id, vendor_name, department) and retain unmatched/null contract links with the asset name and has_contract=false. Inner-join employees/assets and left-join contracts on their IDs; validate that joins neither drop nor multiply assignments. Standard-table primary keys alone are not proof of uniqueness.

Expose the existing seven columns through a thin <LAB_DB>.APP.V_VENDOR_OPPORTUNITY view: contract_id, vendor_name, has_contract, department, inactive_seats, monthly_opportunity, annualized_opportunity (12 x monthly, not a forecast). Grant SELECT on this view to <LAB_ROLE>. Do not change the app API or semantic interface.

Show exact SQL, source checks, refresh-mode/freshness checks and recovery steps. Check for existing targets; do not overwrite them. Wait for my approval before writes. After approval, build only this plan and run sql/v2/02_check_vendor_opportunity.sql after successful refresh; report failures honestly. This is opportunity to review, not realized savings.
```

**Success:** both DTs exist with INCREMENTAL mode, enrichment uses DOWNSTREAM, aggregation has the one-minute freshness target, and the seven-column view remains the app interface. Incremental mode is not a performance guarantee; the first initialization still processes the initial dataset.

**Recovery:** if time is short, review the [structured checkpoint](../sql/v2/checkpoint_vendor_opportunity.sql). This new checkpoint requires isolated live rehearsal before release; do not describe offline validation as refresh proof. It creates fresh objects and fails on collisions rather than replacing an existing pipeline. Inspect a partial build and approve only the missing/repaired steps.

**Terminal / checkpoint writes, only when targets are absent and approved:**

```bash
snow sql --connection "$LAB_CONNECTION" --role ACCOUNTADMIN -D lab_db="$LAB_DB" -D lab_wh="$LAB_WH" -D lab_role="$LAB_ROLE" -f sql/v2/checkpoint_vendor_opportunity.sql
```

**Evidence:** DT modes: ______; last successful refresh: ______; view FQN: ______.

### 3B. Turn staged PDFs into parsed pages

**Goal:** stage directory → stream/task → file log → incremental parsing → incremental pages. The regular file log bridges the stage directory to dynamic tables. No manual clause-text inserts.

**CoCo prompt / plan first:** [canonical document prompts](coco-prompts/build-documents.md).

<!-- prompt: coco-prompts/build-documents.md#1 -->
```text
Read sql/v2/04_document_pipeline.sql and scripts/contract_pipeline.py. Using my lab connection, propose a PDF-only pipeline in <LAB_DB>: stage directory stream -> ingest task -> file log -> INCREMENTAL parsing -> INCREMENTAL pages. Preserve document identity, filename, vendor and one-based page numbers. Use AI_PARSE_DOCUMENT and the existing <LAB_WH>. Do not insert contract prose or overwrite stage files. Show objects, costs and checks; wait for approval. The existing seat-opportunity view stays unchanged.
```

Approve only after reviewing parsing/refresh costs. CoCo builds the plan, or use the supplied checkpoint commands below. Do not run both blindly against differently defined objects.

**Terminal / recovery checkpoint writes:**

```bash
python3 scripts/contract_pipeline.py build --connection "$LAB_CONNECTION"
python3 scripts/contract_pipeline.py backlog --connection "$LAB_CONNECTION"
python3 scripts/contract_pipeline.py import --connection "$LAB_CONNECTION" --group baseline
```

Backlog catches files uploaded before the stream existed. Import verifies unchanged bytes, refreshes the directory, invokes the task and waits for page coverage. The task remains suspended until Search is ready. A wait timeout is not success.

**CoCo prompt / read-only check:**

<!-- prompt: coco-prompts/build-documents.md#2 -->
```text
Inspect the pipeline read-only. Confirm both dynamic tables are INCREMENTAL, every baseline PDF has its expected nonempty page, and no parse error is hidden. Compare Figma's parsed clause with the original PDF. Explain why staged, parsed, and indexed are three different readiness states.
```

**Success:** ten PDFs, expected nonempty pages, preserved filenames/vendor/page numbers, no hidden parsing errors. Search is built later.

**Recovery:** inspect task/DT history and permissions; do not recreate everything, overwrite the PDFs or substitute seeded prose. If a timeout occurs, inspect then retry the bounded `wait` command in the next section.

**Evidence:** parsed baseline count: ______; Figma original versus extracted quote: ______.

## 4. Inspect before you build on top

**Goal:** prove the data contract, not just that objects exist. [Canonical inspection prompt](coco-prompts/inspect-pipeline.md).

<!-- prompt: coco-prompts/inspect-pipeline.md#1 -->
```text
Inspect <LAB_DB>.APP.DT_SEAT_ASSIGNMENTS and DT_VENDOR_OPPORTUNITY read-only. Report refresh_mode, scheduling state and last successful data timestamp. Confirm enrichment feeds aggregation and V_VENDOR_OPPORTUNITY is a thin compatibility view. Run sql/v2/02_check_vendor_opportunity.sql only after both are fresh, with stable raw data and on the same calendar day as refresh. Explain any failure with source rows and propose the smallest fix to the responsible layer; do not mutate anything. Check source uniqueness, join cardinality, grain, unlinked rows, each group's amounts and annualization. Show the top five vendors and contributing departments. Never hardcode expected totals. If stale, propose a leaf refresh for approval rather than silently refreshing or changing business rules.
```

**Terminal / read-only checks, after the freshness gate:**

```bash
snow sql --connection "$LAB_CONNECTION" --role ACCOUNTADMIN -D lab_db="$LAB_DB" -D lab_wh="$LAB_WH" -f sql/v2/02_check_vendor_opportunity.sql
python3 scripts/contract_pipeline.py wait --connection "$LAB_CONNECTION" --group baseline
```

**Success:** all 14 structured checks pass; PDF pages are ready. Capture monthly opportunity and inactive-seat totals to compare with the app. Annualized = monthly × 12, not a forecast. `CURRENT_DATE()` can make eligibility change over time; compare fresh snapshots, not a fixed answer printed in a workbook.

**Recovery:** failed checks need diagnosis. Confirm freshness first, then inspect source keys and per-group differences. Browser Refresh refetches the view; it does not force a dynamic-table refresh. Never hide an error by replacing it with a zero.

**Evidence:** 14 checks: ______; seats: ______; monthly: ______; annualized: ______; snapshot timestamp: ______.

## 5. Deploy the unchanged starter — deployment one

**Goal:** prove the supplied app works before changing it. The starter already contains four KPIs, vendor/department comparisons, review lists, search and contract-linkage filtering. Ask is intentionally a placeholder.

**Terminal / local dependency install and checks:** review the package/lockfile, then run:

```bash
npm --prefix v2/starter-app ci
npm --prefix v2/starter-app test
(cd v2/starter-app && npm exec -- tsc --noEmit)
npm --prefix v2/starter-app run build
```

TypeScript is an explicit gate; the production build alone is not a type check. No global package installation is required here.

**CoCo prompt / local tour:** [canonical prompt](coco-prompts/run-starter-v1.md). Supply your confirmed connection and lab settings. If local auth cannot use your approved CLI configuration, report the blocker; do not add a PAT or alter auth. Cloud verification still uses the app's service identity.

<!-- prompt: coco-prompts/run-starter-v1.md#1 -->
```text
Give me a short tour of v2/starter-app: which files make up the dashboard, the /api/opportunity route and the Ask placeholder; which role and database the app reads with; and what app.yml's `lab` target deploys. Then start it locally with LAB_DB set to my lab database and tell me the local URL. Don't change any files.
```

**CoCo prompt / approve deploy:** [canonical deployment prompts](coco-prompts/deploy-v1.md).

<!-- prompt: coco-prompts/deploy-v1.md#1 -->
```text
Deploy v2/starter-app to my lab account with the default `lab` target in app.yml, using my lab connection. First confirm the account, role, database SEAT_SWEEPER_LAB, warehouse and build integration match what setup created, and that tests, type check and build pass. Show me those settings and wait for my approval. After deploying, record the deploy time, the service version from DESCRIBE APPLICATION SERVICE, and the URL. Don't create new integrations or change grants.
```

Open the returned URL and authenticate. Compare dashboard totals with section 4; confirm search and missing-link filtering work without changing the four KPI totals. A missing contract-register link does not mean a PDF is absent.

<!-- prompt: coco-prompts/deploy-v1.md#2 -->
```text
Inspect my deployed Seat Sweeper app: show its status, version, warehouse and recent logs, and explain which role the app uses to read V_VENDOR_OPPORTUNITY. Read-only.
```

**Success:** deployed dashboard reads your lab view, correct totals, Ask placeholder, recorded service FQN/version/URL.

**Recovery:** inspect build/deployment events and runtime logs. Startup delays are not a reason to create a second service. Keep the first deployment unchanged; save customization for the next step.

**Evidence:** service FQN: ______; version 1: ______; URL: ______; elapsed minutes: ______.

## 6. Make one visible change

**Goal:** add a bounded feature, not a new app. [Canonical personalization prompts](coco-prompts/personalize-v1.md).

<!-- prompt: coco-prompts/personalize-v1.md#1 -->
```text
In v2/starter-app, add a Monthly / Annualized toggle to the vendor opportunity chart in app/page.tsx. Use the existing monthly data and multiply by 12 for the annualized display; label it as annualized opportunity, not a forecast or realized savings. Keep all four KPI totals, details filters and department comparison unchanged. Match the existing visual style, keyboard focus and dark mode. Add tests for both toggle states. Do not change the API, authentication, app.yml, dependencies or other pages. Run tests, explicit type check and build, then show me the diff. Do not deploy yet.
```

**Success:** switching the chart changes its label and values by 12×, with unchanged KPI totals and no API/auth/manifest/dependency changes. Review the diff and test evidence.

**Recovery:** ask CoCo to revert only this exercise's edits after reviewing the diff. Never use a destructive reset or overwrite unrelated work. If time expires, record the incomplete change rather than claiming a successful customization.

**Evidence:** changed files: ______; test result: ______; visible before/after: ______.

## 7. Redeploy that change — deployment two

**Goal:** same service, same URL, a new version with your change.

<!-- prompt: coco-prompts/personalize-v1.md#2 -->
```text
Redeploy my personalized v2/starter-app to the same lab target, connection and application service as deployment one. Confirm the account, service FQN, manifest settings, passing tests/type check/build and the bounded dashboard diff; wait for approval. After approval deploy, record the service version and URL, and verify the Monthly / Annualized toggle at the same URL. Do not create another service or change grants, integrations or authentication.
```

**Success:** verify the toggle in the deployed browser, not only the local preview. Check totals still match the refreshed data.

**Recovery:** if the previous UI remains, inspect actual service version/deploy status before retrying; don't switch targets. If a test failed, fix it before approving deployment.

**Evidence:** version 2: ______; same URL: ______; deployed toggle verified: ______; elapsed minutes: ______.

**Take a 10-minute break.**

## 8. Build and directly test the named agent

**Goal:** prove numeric, document and combined reasoning before connecting Ask.

Review [agent tools SQL](../sql/v2/03_agent_tools.sql) and [agent template](agent/seat_sweeper_agent.agent.yaml.tmpl). This creates `SEAT_SWEEPER_SV` over the opportunity view, `CONTRACT_PDF_SEARCH` over parsed pages and `SEAT_SWEEPER_AGENT` with both tools. Search and Agent calls incur costs. Confirm targets are unused; do not replace an unrelated service or agent.

**Terminal / after approving these writes:**

```bash
python3 scripts/contract_pipeline.py wait --connection "$LAB_CONNECTION" --group baseline
snow sql --connection "$LAB_CONNECTION" --role ACCOUNTADMIN -D lab_db="$LAB_DB" -D lab_wh="$LAB_WH" -D lab_role="$LAB_ROLE" -f sql/v2/03_agent_tools.sql
python3 scripts/contract_pipeline.py wait --connection "$LAB_CONNECTION" --group baseline --search
bash sql/v2/deploy_agent.sh "$LAB_CONNECTION"
python3 scripts/contract_pipeline.py resume --connection "$LAB_CONNECTION" --group baseline
```

The last command resumes ongoing PDF ingestion only after readiness checks. A parsed page and an indexed page are different gates.

**CoCo prompt / read-only inspection:** [canonical build prompt](coco-prompts/build-agent.md).

<!-- prompt: coco-prompts/build-agent.md#1 -->
```text
Inspect what I built in <LAB_DB>.APP, read-only: SEAT_SWEEPER_SV, CONTRACT_PDF_SEARCH, and SEAT_SWEEPER_AGENT. Show their source objects and tool routing. Confirm Search reads DT_CONTRACT_PDF_PAGES rather than RAW.CONTRACTS text. Explain the difference between a linked contract-register row and an indexed PDF. Check app-role access and show one retrieved filename/page/quote.
```

**CoCo prompt / four billable Agent tests:** [canonical test prompt](coco-prompts/use-agent.md). Use the named Agent API through an available supported caller; do not fabricate a direct test or use a substitute SQL chatbot if the caller is unavailable. Record the blocker and resolve it with the facilitator.

<!-- prompt: coco-prompts/use-agent.md#1 -->
```text
Using my lab connection, run these four questions against <LAB_DB>.APP.SEAT_SWEEPER_AGENT, one at a time, and for each show the answer, which tools it called, and any SQL it generated:
1. Which three vendors have the most monthly inactive seat opportunity?
2. What is the cancellation notice period and true-up clause for Figma Enterprise?
3. For the top two vendors by monthly opportunity, how much can we reduce seats this year without penalty?
4. What are the cancellation terms for Zoom Pro?
Then check answer 1 against V_VENDOR_OPPORTUNITY with a read-only query.
```

**Success:** 1 uses `seat_opportunity` and matches fresh view totals; 2 uses `contract_terms` and cites PDF filename/page/quote; 3 uses both and keeps unsupported reduction rights unknown; 4 reports no matching indexed Zoom PDF, not that no agreement exists. Cancellation, nonrenewal and seat reduction are different rights.

**Recovery:** inspect tool routing, Search readiness and app-role grants. Fix only the diagnosed issue with approval. Do not rebuild all resources or infer PDF availability from `has_contract`. If Zoom is already indexed, record that and skip the manufactured before-state.

**Evidence:** agent FQN: ______; tools per question: ______; Figma citation: ______; Zoom before-state: ______.

## 9. Connect Ask and perform deployment three

**Goal:** keep credentials on the server and stream the named Agent response with evidence. Reuse the supplied parser/renderer rather than writing SSE infrastructure. [Canonical Ask prompts](coco-prompts/build-ask.md).

**CoCo prompt / plan:**

<!-- prompt: coco-prompts/build-ask.md#1 -->
```text
In v2/starter-app, plan how to turn the Ask placeholder (app/ask/page.tsx) into a working page backed by my SEAT_SWEEPER_AGENT in <LAB_DB>.APP. Read lib/snowflake.ts and the snowflake-apps skill first.

Requirements:
- A server route POST /api/ask accepts JSON { question }, 1-500 characters, and rejects anything else with 400.
- It calls the named Cortex Agent REST endpoint (/api/v2/databases/{db}/schemas/APP/agents/SEAT_SWEEPER_AGENT:run) with stream: true and Accept: text/event-stream. Reuse the supplied streaming parser/projection and evidence renderer; do not build another parser or use SQL as a fallback.
- Auth uses the app's service token via getRestApiAuthHeader() and the host from getSnowflakeBaseUrl(), with the OAUTH token-type header. No PAT, no secrets, nothing from the browser decides the URL.
- Timeout, safe errors with a correlation ID, and never pass the upstream error body to the browser.
- Show real API activity and elapsed time, a collapsed Thinking panel for API-provided thinking (not signatures), streaming Markdown answers, result tables, PDF citations with filename/page in the source title and quoted excerpts, SQL/query IDs and warnings. No fabricated progress steps. One question at a time. Stop preserves partial output and explains that the upstream agent may still run. Missing completion evidence must remain incomplete.
- Mocked tests for success, bad input, upstream 401, timeout, malformed or interrupted streams, duplicate final events and safe Markdown rendering. Do not log question, answer, thinking or credentials.
- Don't change app.yml, dependencies, the dashboard or /api/opportunity.

Propose the files and tests, then wait for my approval.
```

**CoCo prompt / after approving that plan:**

<!-- prompt: coco-prompts/build-ask.md#2 -->
```text
Implement the plan. Run the type check, tests and build and show me the diff. Don't deploy. Local testing of the live agent needs an OAuth connection, so rely on the mocked tests here; we'll test live after deploying.
```

**Recovery:** preview the supplied completed-Ask checkpoint, inspect the listed file replacements, then approve application. It must preserve your dashboard change. Do not apply it over unrelated edits without reviewing the diff.

```bash
bash v2/checkpoints/apply_completed_ask.sh v2/starter-app
bash v2/checkpoints/apply_completed_ask.sh v2/starter-app --apply
npm --prefix v2/starter-app test
(cd v2/starter-app && npm exec -- tsc --noEmit)
npm --prefix v2/starter-app run build
```

**CoCo prompt / deployment three:** [canonical final deployment prompts](coco-prompts/deploy-v2.md).

<!-- prompt: coco-prompts/deploy-v2.md#1 -->
```text
Perform deployment three: redeploy v2/starter-app with Ask connected to the same `lab` target and service used by deployments one and two. Confirm the target, account and role, that my dashboard personalization is preserved, no runtime egress integration was added, and tests, explicit type check and build pass; then wait for my approval. After deploying, compare the new version in DESCRIBE APPLICATION SERVICE with deployment two and give me the same app URL.
```

**Success:** open `/ask` at the deployed URL and repeat all four questions from section 8. Check streamed output, source disclosure, SQL/result evidence and final completion state. Thinking is shown only if the API supplies it; do not manufacture it. Stop preserves partial text but does not guarantee upstream execution or billing stops. Verify your chart toggle still works.

<!-- prompt: coco-prompts/deploy-v2.md#2 -->
```text
Inspect my deployed app's recent logs, read-only. Find the "op":"ask" lines for my questions and show each one's correlation ID, duration, result and tools. Explain which identity called the agent.
```

**Recovery:** use correlation IDs and safe server logs to diagnose auth/timeout errors. Do not return upstream error bodies or expose service tokens to the browser. A warning or interrupted stream is not a successful completion.

**Evidence:** version 3: ______; same URL: ______; tests/build/types: ______; cited answer: ______; elapsed minutes: ______.

## 10. Import Zoom and ask the identical question again

**Goal:** change the available evidence through the pipeline, with no app redeploy.

Ask **“What are the cancellation terms for Zoom Pro?”** in the deployed app and record the before answer. If Zoom is already indexed from a prior run, report that; do not delete evidence to fabricate the exercise.

Open the held-out [synthetic Zoom PDF](contracts/new-arrival/zoom-pro.pdf). Approve this single-document import/parsing/Search work, then run from the repository root:

```bash
python3 scripts/contract_pipeline.py import --connection "$LAB_CONNECTION" --group new-arrival --search
```

Wait for `Ready`: actual parsed-page coverage **and** Search evidence. The helper uses the same ingestion task and DT pipeline, not a manual text insert. Readiness waits are bounded; a timeout is not an invitation to rerun endlessly or rebuild Search.

Ask the identical Zoom question again. Expand the source and verify `zoom-pro.pdf`, page 1, and the quoted clause against the actual PDF. The fictional cancellation notice is **30 calendar days**. Seat reductions have a separate **10-day notice** and **five-seat minimum**. These are workshop examples, not actual Zoom policies.

**Success:** new answer cites the new PDF; service version and app URL remain unchanged. Zoom still lacks a linked structured contract-register row. Document arrival does not change prices, reconcile linkage or create realized savings. Structured totals can independently age as the 30-day rule advances.

**Recovery:** if parsed but not searchable, inspect Search readiness; if unparsed, inspect ingest task/file log/parse errors. Do not overwrite files, insert clause text, rebuild the agent or redeploy the app as a workaround. Unchanged re-import verifies hashes and skips duplicate bytes.

**Evidence:** before: ______; after + citation: ______; unchanged version: ______; arrival-to-answer minutes: ______.

## 11. Record what worked and decide what to retain

**Goal:** leave a useful result without unattended resources you did not intend to keep.

- Raw tables hold synthetic business facts; structured DTs produce a refreshed opportunity snapshot.
- The PDF task/file log and parsing DTs produce source pages; Search makes them retrievable.
- The semantic view defines numeric meaning; the named agent chooses tools and returns evidence.
- The app provides UI and server-side service authentication. New evidence does not require new UI code.

**Evidence checklist:**

- [ ] Verified Git revision and intended fresh account.
- [ ] Four raw tables and ten staged baseline PDFs.
- [ ] Two structured DTs plus two PDF DTs in INCREMENTAL mode.
- [ ] Fourteen structured checks passed after freshness checks; baseline pages verified.
- [ ] Deployment one: unchanged dashboard and Ask placeholder.
- [ ] Deployment two: same service/URL, visible chart change.
- [ ] Direct agent: numeric, document, combined and missing-document tests recorded.
- [ ] Deployment three: Ask streaming, safe errors and citations; personalization retained.
- [ ] Zoom arrival: cited answer with no fourth deployment.
- [ ] Costs and resource-retention choice reviewed.

**Action / cost review:** ask CoCo to inventory only your lab's warehouse, four DTs, ingest task, Cortex Search service, Agent/semantic view, application service, stage and stored tables. Review current pricing and decide which resources to retain. ACCOUNTADMIN is a workshop convenience, not a production least-privilege design.

**Optional terminal / approved pause of PDF processing only:**

```bash
python3 scripts/contract_pipeline.py pause --connection "$LAB_CONNECTION"
```

That helper suspends the PDF ingest task and its **two PDF dynamic tables only**. It does **not** suspend structured DTs, Search or the application service, and it does not stop storage charges. Warehouse auto-suspend does not prevent resumed work from a scheduled pipeline. Ask CoCo to propose exact scoped suspension/retention actions for the remaining resources and wait for approval; no blanket teardown is supplied here.

**Success:** retained resources and owners are explicit; you know which costs can continue. No unrelated account objects were touched.

**Recovery:** before deleting anything, inspect names, dependencies and desired evidence retention. Destructive cleanup requires separate explicit approval. Never reset the original demo or a shared rehearsal.

**Evidence:** retained: ______; paused: ______; remaining Search/app/storage costs reviewed: ______; cleanup owner/date: ______.

## Facilitator release gate

Before distributing: verify all referenced assets at a published attendee-accessible Git revision; rehearse the new structured checkpoint in an isolated approved account; complete the entire sequence on a fresh account and two timed runs. Record service versions, refresh modes, checks and citations. Offline tests cannot establish billable service readiness or workshop timing. Do not publish personal notebook settings, outputs, local paths or credentials.